# Notebook 04 Inference Chat RAG Tanpa Thinking

Notebook ini khusus untuk inference/chat interaktif. Batch ground truth, scoring, dan plot evaluasi ada di notebook 05.

In [ ]:
# Jalankan sekali di environment baru bila package belum ada
# !pip install -q chromadb rank_bm25 sentence-transformers transformers accelerate bitsandbytes

In [ ]:
import os, re, json, time, pickle, hashlib
from pathlib import Path
from typing import Dict, List, Any

import numpy as np
import torch
import chromadb
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

DATA_PATH = Path("../data/processed_chunks_ringan_pasal_chroma_ready.json")
CHROMA_DB_DIR = Path("../data/chroma_db")
COLLECTION_NAME = "hukum_ketenagakerjaan"
BM25_PATH = Path("../data/bm25_index.pkl")

EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
USE_RERANKER = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("device", DEVICE)
print("data", DATA_PATH.resolve())

In [ ]:
def tokenize_for_bm25(text: str) -> List[str]:
    return re.findall(r"[a-zA-Z0-9_]+|[\u00C0-\u024F\u1E00-\u1EFF]+|[\w]+", str(text).lower())


def normalize_text(text: str) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text


def load_chunks(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"Chunk tidak ditemukan: {path}. Jalankan notebook 01 untuk embedding dan indexing dulu.")
    chunks = json.loads(path.read_text(encoding="utf-8"))
    for i, c in enumerate(chunks[:10]):
        for k in ["id", "text", "display_text", "embedding_text", "citation_text", "metadata"]:
            if k not in c:
                raise ValueError(f"Schema chunk belum sesuai. Field hilang: {k} pada index {i}")
    return chunks

chunks = load_chunks(DATA_PATH)
id_to_chunk = {c["id"]: c for c in chunks}
print("chunks", len(chunks))

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)
client = chromadb.PersistentClient(path=str(CHROMA_DB_DIR))
collection = client.get_collection(COLLECTION_NAME)
print("chroma count", collection.count())
if collection.count() != len(chunks):
    raise ValueError(f"Chroma count {collection.count()} tidak sama dengan chunks {len(chunks)}. Jalankan notebook 01 dulu.")

bm25 = None
bm25_ids = []
if BM25_PATH.exists():
    with BM25_PATH.open("rb") as f:
        payload = pickle.load(f)
    bm25 = payload.get("bm25")
    bm25_ids = payload.get("ids", [])
    print("bm25", len(bm25_ids))
else:
    print("BM25 tidak ditemukan. Dense retrieval tetap berjalan.")

In [ ]:
QUERY_EXPANSIONS = [
    {"triggers": ["pkwt", "kontrak"], "expansion": "perjanjian kerja waktu tertentu kompensasi jangka waktu perpanjangan masa percobaan harian menjadi PKWTT"},
    {"triggers": ["alih daya", "outsourcing"], "expansion": "perusahaan alih daya badan hukum perizinan berusaha perlindungan upah kesejahteraan syarat kerja perselisihan"},
    {"triggers": ["phk", "pesangon", "pengunduran", "mendesak", "surat peringatan"], "expansion": "pemutusan hubungan kerja uang pesangon uang penghargaan masa kerja uang penggantian hak uang pisah pelanggaran bersifat mendesak"},
    {"triggers": ["upah", "pengupahan", "minimum", "tunjangan", "struktur"], "expansion": "upah minimum struktur skala upah upah pokok tunjangan tetap diskriminasi pekerjaan yang sama nilainya"},
    {"triggers": ["lembur", "waktu kerja", "jam kerja"], "expansion": "waktu kerja 7 jam 8 jam 40 jam seminggu perintah kerja lembur persetujuan pekerja"},
]


def expand_query(query: str) -> str:
    q = query.lower()
    additions = []
    for rule in QUERY_EXPANSIONS:
        if any(t in q for t in rule["triggers"]):
            additions.append(rule["expansion"])
    return normalize_text(query + " " + " ".join(additions))


def dense_search(query: str, fetch_k: int = 40) -> List[Dict[str, Any]]:
    q_emb = embedding_model.encode(["query: " + query], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)[0].tolist()
    res = collection.query(query_embeddings=[q_emb], n_results=fetch_k, include=["documents", "metadatas", "distances"])
    hits = []
    for doc_id, doc, meta, dist in zip(res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({"id": doc_id, "text": doc, "metadata": meta or {}, "dense_distance": float(dist), "source": "dense"})
    return hits


def bm25_search(query: str, fetch_k: int = 40) -> List[Dict[str, Any]]:
    if bm25 is None or not bm25_ids:
        return []
    scores = bm25.get_scores(tokenize_for_bm25(query))
    order = np.argsort(scores)[::-1][:fetch_k]
    ids = [bm25_ids[i] for i in order if scores[i] > 0]
    if not ids:
        return []
    got = collection.get(ids=ids, include=["documents", "metadatas"])
    lookup = {doc_id: (doc, meta) for doc_id, doc, meta in zip(got["ids"], got["documents"], got["metadatas"])}
    hits = []
    for i in order:
        doc_id = bm25_ids[i]
        if scores[i] <= 0 or doc_id not in lookup:
            continue
        doc, meta = lookup[doc_id]
        hits.append({"id": doc_id, "text": doc, "metadata": meta or {}, "bm25_score": float(scores[i]), "source": "bm25"})
    return hits


def rrf_fuse(result_sets: List[List[Dict[str, Any]]], weights=None, rrf_k: int = 60) -> List[Dict[str, Any]]:
    weights = weights or [1.0] * len(result_sets)
    fused = {}
    for hits, weight in zip(result_sets, weights):
        for rank, hit in enumerate(hits, 1):
            item = fused.setdefault(hit["id"], {"score": 0.0, "hit": hit})
            item["score"] += weight / (rrf_k + rank)
            item["hit"].update({k: v for k, v in hit.items() if k not in item["hit"]})
    out = []
    for item in sorted(fused.values(), key=lambda x: x["score"], reverse=True):
        hit = item["hit"]
        hit["rrf_score"] = float(item["score"])
        out.append(hit)
    return out

reranker = None
if USE_RERANKER:
    from sentence_transformers import CrossEncoder
    reranker = CrossEncoder(RERANKER_MODEL_NAME, device=DEVICE)
    print("reranker aktif", RERANKER_MODEL_NAME)


def rerank_documents(query: str, docs: List[Dict[str, Any]], k: int = 8) -> List[Dict[str, Any]]:
    if reranker is None or not docs:
        return docs[:k]
    scores = reranker.predict([(query, d["text"]) for d in docs])
    for d, s in zip(docs, scores):
        d["rerank_score"] = float(s)
    return sorted(docs, key=lambda x: x.get("rerank_score", 0.0), reverse=True)[:k]


def dedupe_legal_hits(docs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen, out = set(), []
    for d in docs:
        m = d.get("metadata", {})
        key = (m.get("source_file", ""), m.get("pasal_id", ""), m.get("chunk_index", ""), d.get("id", ""))
        if key in seen:
            continue
        seen.add(key)
        out.append(d)
    return out


def retrieve_documents(query: str, fetch_k: int = 40, final_k: int = 8) -> List[Dict[str, Any]]:
    qx = expand_query(query)
    fused = rrf_fuse([dense_search(qx, fetch_k), bm25_search(qx, fetch_k)], weights=[1.0, 0.8])
    fused = dedupe_legal_hits(fused)
    return rerank_documents(query, fused, k=final_k)


def build_reference(meta: Dict[str, Any]) -> str:
    citation = meta.get("citation_text") or meta.get("citation") or ""
    source = meta.get("source_file") or meta.get("file_name") or ""
    pasal = meta.get("pasal_id") or meta.get("article") or ""
    parts = [citation, source, pasal]
    return " | ".join([str(p) for p in parts if str(p).strip()])


def assemble_context(docs: List[Dict[str, Any]], max_docs: int = 6) -> str:
    blocks = []
    for i, d in enumerate(docs[:max_docs], 1):
        meta = d.get("metadata", {})
        ref = build_reference(meta) or f"Dokumen {i}"
        text = normalize_text(d.get("text", ""))[:2500]
        blocks.append(f"[R{i}] {ref}\n{text}")
    return "\n\n".join(blocks)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = os.getenv("RAG_LLM_MODEL_ID", "Qwen/Qwen3.5-9B")
MAX_NEW_TOKENS = int(os.getenv("MAX_NEW_TOKENS", "700"))
USE_4BIT = os.getenv("USE_4BIT", "1") == "1" and torch.cuda.is_available()

quantization_config = None
if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    quantization_config=quantization_config,
    trust_remote_code=True,
)
model.eval()
print("model ready", MODEL_ID)

In [ ]:
def strip_thinking(text: str) -> str:
    text = re.sub(r"<think>[\s\S]*?</think>", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^\s*(analysis|reasoning)\s*:\s*", "", text, flags=re.IGNORECASE)
    return text.strip()


def apply_chat_template_no_thinking(messages: List[Dict[str, str]]) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        except TypeError:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return "\n".join([f"{m['role']}: {m['content']}" for m in messages]) + "\nassistant:"


def generate_chat_text(messages: List[Dict[str, str]], max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    prompt = apply_chat_template_no_thinking(messages)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(new_ids, skip_special_tokens=True)
    return strip_thinking(text)

SYSTEM_PROMPT = """Kamu adalah asisten hukum ketenagakerjaan Indonesia. Jawab hanya berdasarkan REFERENSI yang diberikan. Jika referensi tidak cukup, katakan bahwa referensi yang tersedia belum cukup. Jangan tampilkan proses berpikir. Sebutkan dasar hukum natural seperti PP Nomor 35 Tahun 2021 Pasal 52 ayat 2. Jawaban harus ringkas, jelas, dan langsung menjawab kasus."""


def generate_answer(question: str, k: int = 8, max_context_docs: int = 6) -> Dict[str, Any]:
    t0 = time.time()
    docs = retrieve_documents(question, final_k=k)
    context = assemble_context(docs, max_docs=max_context_docs)
    user_prompt = f"REFERENSI:\n{context}\n\nPERTANYAAN:\n{question}\n\nJAWABAN:"
    answer = generate_chat_text([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ])
    latency = time.time() - t0
    refs = []
    for i, d in enumerate(docs, 1):
        meta = d.get("metadata", {})
        refs.append({
            "rank": i,
            "chunk_id": d.get("id", ""),
            "reference": build_reference(meta),
            "source_file": meta.get("source_file", ""),
            "pasal_id": meta.get("pasal_id", ""),
            "rrf_score": d.get("rrf_score", None),
            "rerank_score": d.get("rerank_score", None),
            "text_preview": normalize_text(d.get("text", ""))[:500],
        })
    return {"question": question, "answer": answer, "references": refs, "latency_seconds": latency}

In [ ]:
question = "Apa hak pekerja jika di-PHK?"
out = generate_answer(question, k=8, max_context_docs=6)

print("PERTANYAAN")
print(question)
print("\nJAWABAN")
print(out["answer"])
print("\nREFERENSI")
for r in out["references"][:6]:
    print(f"{r['rank']}. {r['reference']}")
print(f"\nLatency: {out['latency_seconds']:.2f} detik")

In [ ]:
while True:
    q = input("Tanya RAG hukum (exit untuk berhenti): ").strip()
    if q.lower() in {"exit", "quit", "q"}:
        break
    if not q:
        continue
    out = generate_answer(q, k=8, max_context_docs=6)
    print("\nJAWABAN")
    print(out["answer"])
    print("\nREFERENSI")
    for r in out["references"][:6]:
        print(f"{r['rank']}. {r['reference']}")
    print(f"\nLatency: {out['latency_seconds']:.2f} detik\n")